# 04 - PLS variants

**Purpose**: PLSCanonical fits between connectivity (X-block) and
kinematics (Y-block) for each of three research questions:

- ``injury_snapshot`` -- kinematic profile at Post_Injury_2-4.
- ``deficit_delta``  -- change from Baseline to Post_Injury_2-4.
- ``recovery_delta`` -- change from Post_Injury_2-4 to Post_Rehab_Test.

Set ``VARIANT`` in the parameters cell to pick which one to run, or
leave it as ``'all'`` to run them sequentially.

**Input**: ``data.FCDGdf_wide`` and ``data.AKDdf_agg_contact_flat()``
restricted to the important regions / features identified in 01 and 02.


In [ ]:
# parameters
VARIANT = 'all'  # 'injury_snapshot', 'deficit_delta', 'recovery_delta', or 'all'
N_COMPONENTS = 2
TOP_N = 15       # Top connectivity regions to label per LV


In [ ]:
import pandas as pd                                                                          # dataframes
import matplotlib.pyplot as plt                                                              # plotting

from endpoint_ck_analysis.config import CACHE_DIR, EXAMPLE_OUTPUT_DIR                        # cache dir for the parquet files we read; example_output dir for saved figures
from endpoint_ck_analysis.data_loader import load_all                                        # one-shot loader
from endpoint_ck_analysis.helpers.dimreduce import build_y_phase, build_y_shift, run_pls     # Y-block builders (snapshot vs phase-delta) + PLS fitter
from endpoint_ck_analysis.helpers.plotting import plot_pls, stamp_version                    # PLS three-panel figure helper + figure version footer

EXAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)                                        # ensure output folder exists
data = load_all()                                                                            # load (uses cache from 00_setup)
agg_flat = data.AKDdf_agg_contact_flat()                                                     # flatten the multi-index aggregated kinematics so we can filter on phase + contact_group as columns

# Pull the important regions / features written by notebooks 01 and 02.
important_regions = pd.read_parquet(CACHE_DIR / 'important_regions.parquet')['region_hemi'].tolist() # parquet was written by notebook 01; .tolist() converts the column into a Python list of strings
important_features = pd.read_parquet(CACHE_DIR / 'important_features.parquet')['feature'].tolist()    # parquet from notebook 02; same shape

X_block = data.FCDGdf_wide[important_regions].fillna(0)                                      # X-block for PLS: subjects x important connectivity regions; fillna(0) since PLS doesn't accept NaNs


## 1. Build the three Y-blocks

These share the same X-block (connectivity) so results compare directly.


In [ ]:
Y_injury = build_y_phase(agg_flat, important_features, 'Post_Injury_2-4')                    # snapshot Y: subject x feature kinematics at the post-injury phase
Y_deficit = build_y_shift(agg_flat, important_features, 'Baseline', 'Post_Injury_2-4')        # delta Y: how much each kinematic feature shifted from baseline to post-injury (per subject)
Y_recovery = build_y_shift(agg_flat, important_features, 'Post_Injury_2-4', 'Post_Rehab_Test') # delta Y: how much each feature recovered from post-injury to post-rehab

Y_BLOCKS = {                                                                                  # dispatch table: maps the variant string to (Y_block, label) pair so the loop below can pick which to run
    'injury_snapshot': (Y_injury, 'Injury snapshot (Post_Injury_2-4 kinematics)'),
    'deficit_delta':   (Y_deficit, 'Injury deficit (Post_Injury_2-4 - Baseline)'),
    'recovery_delta':  (Y_recovery, 'ABT recovery (Post_Rehab_Test - Post_Injury_2-4)'),
}


## 2. Run the selected variant(s)

If ``VARIANT == 'all'`` every variant runs in order; otherwise only the
selected one. Each call produces a three-panel figure per LV
(X-loadings, Y-loadings, score-vs-score scatter).


In [ ]:
variants = list(Y_BLOCKS.keys()) if VARIANT == 'all' else [VARIANT]                       # if VARIANT='all', loop over all three; otherwise just the chosen one
if VARIANT != 'all' and VARIANT not in Y_BLOCKS:                                          # guard against typos in the parameters cell
    raise ValueError(f"Unknown VARIANT {VARIANT!r}. Choose one of {list(Y_BLOCKS) + ['all']}.")

results = {}                                                                              # variant name -> dict returned by run_pls (contains pls model, X_scores, Y_scores, X_loadings, Y_loadings, subjects, label)
for variant in variants:
    Y, label = Y_BLOCKS[variant]                                                          # tuple unpack: Y is the Y-block dataframe, label is the human-readable string
    print(f'\n=== {variant} ===')                                                        # section divider in the printed output
    results[variant] = run_pls(X_block, Y, n_components=N_COMPONENTS, label=label)        # fit PLSCanonical between connectivity X-block and this variant's Y-block
    plot_pls(                                                                              # render the three-panel figure (X-loadings, Y-loadings, cross-score scatter) per LV
        results[variant], top_n=TOP_N,                                                     # top_n caps how many connectivity loadings to label
        save_dir=EXAMPLE_OUTPUT_DIR, slug=f"04_pls_{variant}",                             # also persist each LV figure to example_output/04_pls_{variant}_{LV}.png so the gallery (notebook 99) can show it later without re-running 04
    )


**What you just saw.** Three figures per variant (one per latent
variable). Each figure has three panels:
- **Left (X-loadings):** which residual-connectivity regions drive
  this LV. Long bar = the region's residual connectivity is part of
  the coupling axis.
- **Middle (Y-loadings):** which kinematic features drive this LV.
  Long bar = the feature's variation is part of the coupling axis.
- **Right (cross-score scatter):** each subject as a point at
  (X-side score, Y-side score). The dashed line is a least-squares
  fit; r and p in the title quantify how well the two sides line up.

**Cross-score scatter -- the headline.** A clean diagonal scatter
with high r (>0.8) means the X-block and Y-block share a real
coupled axis: subjects who score high on the connectivity-side LV
also score high on the kinematic-side LV. *Example:* a clean
positive scatter on LV1 of `recovery_delta` would mean "mice with
more residual connectivity on the LV1 axis recovered more reaching
capacity post-ABT" -- the headline scientific claim of this
analysis. A spread cloud with low r means no coherent coupling
was found at this LV.

**Top vs low loadings -- what they mean.** A region with a long
positive bar on the X side AND a feature with a long positive bar
on the Y side covary positively. Same signs across blocks =
positive coupling; opposite signs = inverse coupling. A near-zero
bar means the region/feature isn't contributing to this LV.

**Variant-specific interpretation.**
- *Injury snapshot:* X-loadings = which residual-connectivity
  regions go with which post-injury reaching style. Heads up the
  "given what residual connectivity each mouse had, here's the
  reaching style it produces" story.
- *Deficit delta:* X-loadings = which residual-connectivity
  regions track the magnitude/direction of the immediate kinematic
  shift baseline -> post-injury. The regions whose residual
  connectivity best predicts the deficit.
- *Recovery delta:* X-loadings = which residual-connectivity
  regions track post-ABT change in kinematics. The regions whose
  residual connectivity best predicts ABT response. This is the
  headline "does residual connectivity explain differential ABT
  response" question.

**At small cohort sizes, every PLS will look highly correlated by
construction.** With few subjects and many features per side, PLS
has enough freedom to always find a fit. Treat the cross-score r
as descriptive (here's what the data shows), NOT inferential
(there's a real underlying relationship). At larger N this
becomes a real test.


## 3. Export latent-variable scores for the gallery

Each variant's subject scores are written to the cache so the figure
gallery can assemble a summary figure without re-fitting.


In [ ]:
for variant, r in results.items():                                                         # iterate over the variants that ran (depending on VARIANT setting, this is 1 or 3)
    out = pd.DataFrame(r['X_scores'], index=r['subjects'],                                 # subjects' positions on each LV in connectivity-side latent space
                       columns=[f'LV{i+1}' for i in range(r['X_scores'].shape[1])])       # LV1, LV2, ... column names; .shape[1] = number of latent variables actually fit
    out.to_parquet(CACHE_DIR / f'pls_{variant}_X_scores.parquet')                          # one parquet per variant; figure gallery / future analyses can read these without re-fitting
    print(f'Wrote pls_{variant}_X_scores.parquet, N={len(out)}')                           # status; N here is matched-subject count for this variant
